In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
import logging
import warnings
from joblib import Parallel, delayed
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [4]:

# Load data to double check exact values and column types
df = pd.read_csv("Cleaned_Environment_Temperature_Data.csv")

print("Columns:", df.columns.tolist())
print("Areas:", len(df['Area'].unique()))
print("Months unique:", df['Months'].unique())

# Clean seasonal names
season_clean = {
    'Dec\x96Jan\x96Feb': 'Winter',
    'Mar\x96Apr\x96May': 'Spring',
    'Jun\x96Jul\x96Aug': 'Summer',
    'Sep\x96Oct\x96Nov': 'Autumn'
}

df['Months_Clean'] = df['Months'].replace(season_clean)
print("Cleaned Months unique:", df['Months_Clean'].unique())

Columns: ['Area', 'Months', 'Year', 'Unit', 'Standard_Deviation', 'Temperature_Change']
Areas: 245
Months unique: <StringArray>
[              'April',              'August',            'December',
         'DecJanFeb',            'February',             'January',
                'July',                'June',         'JunJulAug',
               'March',         'MarAprMay',                 'May',
 'Meteorological year',            'November',             'October',
           'September',         'SepOctNov']
Length: 17, dtype: str
Cleaned Months unique: <StringArray>
[              'April',              'August',            'December',
              'Winter',            'February',             'January',
                'July',                'June',              'Summer',
               'March',              'Spring',                 'May',
 'Meteorological year',            'November',             'October',
           'September',              'Autumn']
Length: 17, dtype:

In [ ]:
# Suppress warnings and logs
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
warnings.filterwarnings('ignore')

def evaluate_model(area, category, df_subset):
    """Worker function to train on 90%, test on 10%, and return error metrics."""
    try:
        # 1. Format the Data
        prophet_df = pd.DataFrame()
        prophet_df['ds'] = pd.to_datetime(df_subset['Year'].astype(str) + '-01-01')
        prophet_df['y'] = df_subset['Temperature_Change'].values
        
        # 2. Implement the 90/10 Split
        # 1961 to 2013 (53 years, ~90%)
        train_data = prophet_df[prophet_df['ds'].dt.year <= 2013]
        # 2014 to 2019 (6 years, ~10%)
        test_data = prophet_df[prophet_df['ds'].dt.year > 2013]
        
        if len(train_data) < 4 or test_data.empty:
            return None
            
        # 3. Train the isolated model purely on the past (1961 - 2013)
        model = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
        model.fit(train_data)
        
        # 4. Ask the model to predict the holdout dates (2014 - 2019)
        future = test_data[['ds']]
        forecast = model.predict(future)
        
        # 5. Extract actual vs predicted values
        y_actual = test_data['y'].values
        y_pred = forecast['yhat'].values
        
        # 6. Calculate grading metrics
        mae = mean_absolute_error(y_actual, y_pred)
        rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
        
        # Using R-Squared on small time-series holdouts can be unstable, 
        # but we capture it to check for severe trend failures
        r2 = r2_score(y_actual, y_pred) 
        
        return {
            'Area': area,
            'Category': category,
            'Test_Size_Years': len(test_data),
            'MAE': mae,
            'RMSE': rmse,
            'R-Squared': r2
        }
        
    except Exception as e:
        return None

if __name__ == "__main__":
    print("Loading data for the 10% Backtest Evaluation...")
    df = pd.read_csv("Cleaned_Environment_Temperature_Data.csv")
    
    season_mapping = {
        'Dec\x96Jan\x96Feb': 'Winter', 'Mar\x96Apr\x96May': 'Spring',
        'Jun\x96Jul\x96Aug': 'Summer', 'Sep\x96Oct\x96Nov': 'Autumn'
    }
    df['Months'] = df['Months'].replace(season_mapping)
    
    unique_categories = df['Months'].unique()
    all_areas = df['Area'].unique()
    
    # Filter for the 186 pristine areas we established earlier
    clean_areas = []
    for area in all_areas:
        area_df = df[df['Area'] == area]
        is_complete = True
        for category in unique_categories:
            if len(area_df[area_df['Months'] == category]['Year'].unique()) < 59:
                is_complete = False
                break
        if is_complete:
            clean_areas.append(area)
            
    # Slice the dataframe into testing tasks
    tasks = []
    for area in clean_areas:
        for category in unique_categories:
            subset = df[(df['Area'] == area) & (df['Months'] == category)]
            tasks.append((area, category, subset))
                
    print(f"Deploying holdout test on {len(tasks)} isolated Prophet models...")
    
    # Run the backtests in parallel across all CPU cores
    results = Parallel(n_jobs=-1)(
        delayed(evaluate_model)(area, cat, data) 
        for area, cat, data in tqdm(tasks, desc="Evaluating Matrix")
    )
    
    valid_results = [r for r in results if r is not None]
    evaluation_df = pd.DataFrame(valid_results)
    
    # Calculate global matrix accuracy
    print("\n=== MATRIX EVALUATION RESULTS ===")
    print(f"Global Average MAE:  {evaluation_df['MAE'].mean():.4f} °C")
    print(f"Global Average RMSE: {evaluation_df['RMSE'].mean():.4f} °C")
    
    # Export the report card
    evaluation_df.to_csv("Matrix_10Percent_Evaluation_Report.csv", index=False)
    print("\nDetailed grade-book saved to 'Matrix_10Percent_Evaluation_Report.csv'")

Loading data for the 10% Backtest Evaluation...
Deploying holdout test on 3162 isolated Prophet models...


Evaluating Matrix: 100%|██████████| 3162/3162 [01:32<00:00, 34.21it/s]



=== MATRIX EVALUATION RESULTS ===
Global Average MAE:  0.6136 °C
Global Average RMSE: 0.7340 °C

Detailed grade-book saved to 'Matrix_10Percent_Evaluation_Report.csv'
